In [10]:
from pathlib import Path
from typing import Final

import numpy as np
import numpy.typing as npt
import pandas as pd
import polars as pl
import seaborn as sb
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

pl.Config.set_tbl_cols(50)
pl.Config.set_tbl_rows(45)


polars.config.Config

In [2]:
PROJECT_ROOT: Path = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT:Path = PROJECT_ROOT.parent

DATA_DIR: Final[Path] = PROJECT_ROOT / "data" / "raw"
TRAIN_FRAC: Final[float] = 0.7

houses: list[str] = [
    "csh101", "csh102", "csh103", "csh104", "csh105", "csh106", "csh107", "csh108", "csh109", "csh110",
    "csh111", "csh112", "csh113", "csh114", "csh115", "csh116", "csh117", "csh118", "csh119", "csh120",
    "csh121", "csh122", "csh123", "csh124", "csh125", "csh126", "csh127", "csh128", "csh129", "csh130",
]



In [3]:
data_files: list[Path] = [DATA_DIR / house / f"{house}.ann.features.csv" for house in houses]
print(f"Found {len(data_files)} files")
print(data_files[:3])


Found 30 files
[PosixPath('/home/treyt/Documents/BYUI/2026/machine-learning/uv-final-project/data/raw/csh101/csh101.ann.features.csv'), PosixPath('/home/treyt/Documents/BYUI/2026/machine-learning/uv-final-project/data/raw/csh102/csh102.ann.features.csv'), PosixPath('/home/treyt/Documents/BYUI/2026/machine-learning/uv-final-project/data/raw/csh103/csh103.ann.features.csv')]


# Helper Functions

In [4]:
def normalize_weirdos(name: str) -> str:
    if name.startswith("r1.") or name.startswith("r2."):
        return name[3:]
    return name


In [ ]:
def generalize_labels(specific_label:str) -> str:
    match specific_label:
        case 

# Data Preprocessing

In [ ]:
df_all: pl.LazyFrame = (
    pl.scan_csv(data_files)
    .with_columns(pl.col("activity").str.replace(r"^r[12]\.", "").alias("activity")) # reformat odd `r1.sleep` `r2.Dress` labels
    .filter(pl.col('activity') != "Other_Activity")
)

class_counts: pl.LazyFrame = (
    df_all
    .group_by("activity")
    .len()
    .sort("len", descending=True)
)

#class_counts.filter(pl.col("activity").str.contains("Dress|Breakfast|Hygiene|Sleep")).collect()
class_counts.collect()


activity,len
str,u32
"""Entertain_Guests""",1274082
"""Personal_Hygiene""",951351
"""Cook_Breakfast""",615702
"""Watch_TV""",596557
"""Sleep""",567863
"""Work_On_Computer""",551722
"""Dress""",505631
"""Groom""",394545
"""Cook_Dinner""",375098


In [13]:
classes: pl.LazyFrame = df_all.select(pl.col('activity')).unique()

classes.collect().sort(by=pl.col('activity'), descending=False)

activity
str
"""Bathe"""
"""Bed_Toilet_Transition"""
"""Cook"""
"""Cook_Breakfast"""
"""Cook_Dinner"""
"""Cook_Lunch"""
"""Dress"""
"""Drink"""
"""Eat"""


In [ ]:
general_map:dict[str,str] = {
    "Bathe": "Hygiene",
    "Personal_Hygiene": "Hygiene",
    "Groom": "Hygiene",
    "Toilet": "Hygiene",
    "Bed_Toilet_Transition": "Hygiene",
    "Dress": "Hygiene",

    "Cook": "Meal_Prep",
    "Cook_Breakfast": "Meal_Prep",
    "Cook_Lunch": "Meal_Prep",
    "Cook_Dinner": "Meal_Prep",

    "Wash_Dishes": "Meal_Cleanup",
    "Wash_Breakfast_Dishes": "Meal_Cleanup",
    "Wash_Lunch_Dishes": "Meal_Cleanup",
    "Wash_Dinner_Dishes": "Meal_Cleanup",

    "Eat": "Eating",
    "Drink": "Eating",
    "Eat_Breakfast": "Eating",
    "Eat_Lunch": "Eating",
    "Eat_Dinner": "Eating",

    "Sleep": "Sleep_Rest",
    "Go_To_Sleep": "Sleep_Rest",
    "Wake_Up": "Sleep_Rest",
    "Nap": "Sleep_Rest",
    "Sleep_Out_Of_Bed": "Sleep_Rest",

    "Work": "Work_Study",
    "Work_On_Computer": "Work_Study",
    "Work_At_Desk": "Work_Study",
    "Work_At_Table": "Work_Study",

    "Exercise": "Exercise",

    "Read": "Leisure",
    "Phone": "Leisure",
    "Relax": "Leisure",
    "Watch_TV": "Leisure",
    "Entertain_Guests": "Leisure",

    "Morning_Meds": "Medication",
    "Evening_Meds": "Medication",
    "Take_Medicine": "Medication",

    "Enter_Home": "Home_Transition",
    "Leave_Home": "Home_Transition",
    "Step_Out": "Home_Transition",

    "Laundry": "Household"
}

df = df.with_columns(
    pl.col("activity")
      .replace(general_map)
      .alias("activity_general")
)

In [ ]:
cv: StratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf: XGBClassifier = XGBClassifier(
    random_state=42,
    eval_metric="mlogloss",
    n_estimators=400,
    learning_rate=0.09,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    verbosity=1,
)


In [ ]:
f1: npt.NDArray[np.float64] = cross_val_score(
    clf,
    X,
    y,
    cv=cv,
    scoring=make_scorer(f1_score, average="weighted"),
    n_jobs=-1,
)
print("5-fold weighted-F1: %.3f ± %.3f" % (f1.mean(), f1.std()))


In [ ]:
model: XGBClassifier = xgb.XGBClassifier(random_state=42, eval_metric="mlogloss", n_jobs=-1)
model.fit(X_train, y_train)


In [ ]:
y_pred: npt.NDArray[np.int64] = model.predict(X_test)
y_proba: npt.NDArray[np.float64] = model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 score:", f1_score(y_test, y_pred, average="weighted"))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

if y_proba.shape[1] == 2:
    roc: float = roc_auc_score(y_test, y_proba[:, 1])
else:
    roc = roc_auc_score(y_test, y_proba, multi_class="ovr")
print("ROC AUC:", roc)

precision: npt.NDArray[np.float64]
recall: npt.NDArray[np.float64]
precision, recall, _ = precision_recall_curve(y_test == 1, y_proba[:, 1])
pr_auc: float = auc(recall, precision)
print("PR AUC for class 1:", pr_auc)
